In [1]:
import numpy as np
from evaluate import load
from sentence_transformers import SentenceTransformer
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

d:\_SELF_MASTERs\_UDACITY_MASTERS\env_python\pytorch_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# EM

In [2]:
# Let's compare predicted fruit names with the correct labels.
preds = ["Apple", "banana ", " Orange"]
labels = ["apple", "banana", "grape"]

def normalize(s: str) -> str:
    return s.lower().strip()

def exact_match(pred: str, label: str) -> int:
    return int(normalize(pred) == normalize(label))

em_scores = [exact_match(p, l) for p, l in zip(preds, labels)]

em_accuracy = sum(em_scores) / len(em_scores)

print(f"Individual Scores: {em_scores}")
print(f"Average Exact Match Accuracy: {em_accuracy:.2f}")

Individual Scores: [1, 1, 0]
Average Exact Match Accuracy: 0.67


# Lexical Similarity (ROUGE)


In [3]:
# Define a prediction and a reference text
pred = "the quick brown fox"
label = "the fox is quick and brown"

# Load the ROUGE metric from the 'evaluate' library
rouge = load("rouge")

results = rouge.compute(predictions=[pred], references=[label])

print(f"ROUGE-1 Score: {results['rouge1']:.4f}")
print(f"ROUGE-L Score: {results['rougeL']:.4f}")

ROUGE-1 Score: 0.8000
ROUGE-L Score: 0.6000


# Semantic Similarity

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

labels = ["A dog is a loyal pet", "Cats are independent animals", "The sky is blue"]
preds = [
    "Dogs make great companions",
    "A cat is a solitary creature",
    "The ocean is vast",
]

pred_embeddings = model.encode(preds)
label_embeddings = model.encode(labels)

for i in range(len(preds)):
    similarity = np.dot(pred_embeddings[i], label_embeddings[i]) / (
        np.linalg.norm(pred_embeddings[i]) * np.linalg.norm(label_embeddings[i])
    )
    print(
        f"Pair {i + 1}:\n  Pred:  '{preds[i]}'\n  Label: '{labels[i]}'\n  Similarity: {similarity:.4f}\n"
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3103.32it/s]


Pair 1:
  Pred:  'Dogs make great companions'
  Label: 'A dog is a loyal pet'
  Similarity: 0.6147

Pair 2:
  Pred:  'A cat is a solitary creature'
  Label: 'Cats are independent animals'
  Similarity: 0.6848

Pair 3:
  Pred:  'The ocean is vast'
  Label: 'The sky is blue'
  Similarity: 0.3098



# LLM-as-a-Judge


In [ ]:
from litellm import completion

RUBRIC = """
Score 1.0 if the predicted animal is the same as the label.
Score 0.5 if the prediction is a different animal but from the same biological class (e.g., both are mammals).
Score 0.0 otherwise (e.g., a mammal and a reptile).
"""
def llm_as_judge(pred,label,rubric):
    SYSTEM_PROMPT = f"""You are an expert evaluator. Use the following rubric to score the prediction. Format your response as:
    <reasoning>...</reasoning>
    <score>FLOAT_ANSWER</score>
    where FLOAT_ANSWER is a float between 0 and 1.
    RUBRIC:
    {rubric}
    """
    USER_PROMPT = f"prediction {pred} for label {label}"
    
    response = completion(
    model="ollama/deepseek-r1:8b",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": USER_PROMPT
        }
    ],
    api_base="http://localhost:11434"
    )
    return response.choices[0].message.content

score1 = llm_as_judge(pred="Lion", label="Lion", rubric=RUBRIC)
print(f"--> Final Score:\n {score1}\n")

score2 = llm_as_judge(pred="Tiger", label="Lion", rubric=RUBRIC)
print(f"--> Final Score:\n {score2}\n")

score3 = llm_as_judge(pred="Snake", label="Lion", rubric=RUBRIC)
print(f"--> Final Score:\n {score3}\n")

23:33:17 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
23:33:18 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


--> Final Score: <reasoning>The prediction "Lion" exactly matches the label "Lion". According to the rubric, this qualifies for a score of 1.0.</reasoning>
<score>1.0</score>

--> Final Score: <reasoning>Tiger and Lion are both members of the Felidae family, making them part of the same biological class (mammals). However, they are different species, so the prediction does not exactly match the label. According to the rubric, this qualifies for a score of 0.5.</reasoning>
<score>0.5</score>

